# Explainable AI (XAI)
- **Occlusion Sensitivity Heatmaps**:
    - Occlusion tests the model’s response when local patches are masked, producing an importance map over the image.
    - It is useful for texture-driven models because it reveals whether the classifier depends on local texture regions instead of only global statistics.
    - Advantages: highlights regions that affect the decision, helps detect whether the model is using meaningful defect texture, and provides a sanity-check for trust and debugging.
    - Limitations: patch-based heatmaps are coarse, can be noisy for texture models, and may reflect correlated texture patterns rather than exact physical defect shapes.
    - Practical caveat: the attention may appear patchy and texture-driven, not aligned to consistent defect shapes, so use heatmaps as diagnostic guidance rather than precise localization.

#TODO interpret model using generated heatmaps 

In [ ]:
from pathlib import Path

from attr import dataclass
from matplotlib import pyplot as plt
from matplotlib.image import imread
import numpy as np
from skimage.transform import resize
from steelblast_classic_features import CLASS_NAMES, LABELS

import shutil


@dataclass(frozen=True)
class HeatmapConfig:
    heatmap_dir: Path = Path("steelblast_svm_glcm_dwt_focus_heatmaps")
    image_size: int = 256
    occlusion_patch_size: int = 32
    occlusion_stride: int = 16
    max_per_case: int | None = 5
    occlusion_fill_mode: str = "median"  # median or zero
    clear_output_dir: bool = True





def read_display_image(image_path: Path, image_size: int) -> np.ndarray:
    image = imread(image_path)

    if image.ndim == 2:
        image = np.repeat(image[..., np.newaxis], 3, axis=2)
    else:
        image = image[..., :3]

    image = resize(
        image,
        (image_size, image_size),
        anti_aliasing=True,
        preserve_range=True,
    )

    image = image.astype(np.float32)
    if image.max() > 1.0:
        image /= 255.0

    return np.clip(image, 0.0, 1.0)


def save_focus_pair(
    display_image: np.ndarray,
    heatmap: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), constrained_layout=True)
    axes[0].imshow(display_image)
    axes[0].set_title("Example image")
    axes[0].set_axis_off()

    axes[1].imshow(display_image)
    vmax = float(heatmap.max()) if heatmap.size and heatmap.max() > 0 else 1.0
    overlay = axes[1].imshow(heatmap, cmap="jet", alpha=0.62, vmin=0, vmax=vmax)
    axes[1].set_title("Activation heatmap")
    axes[1].set_axis_off()
    fig.colorbar(overlay, ax=axes[1], label="Decision-score drop")
    fig.suptitle(title, fontsize=16)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)

# this function computes an occlusion-based heatmap for a given image and predicted label. It systematically occludes patches of the image and measures the change in the model's confidence score for the predicted label. The resulting heatmap highlights areas of the image that are important for the model's decision-making process.
def compute_occlusion_focus_heatmap(
    image: np.ndarray,
    predicted_label: int,
    score_fn,
    patch_size: int,
    stride: int,
    fill_value: float,
) -> np.ndarray:
    """Generic occlusion engine decoupled from model internals."""
    base_confidence = score_fn(image, predicted_label)
    height, width = image.shape[:2]
    heatmap = np.zeros((height, width), dtype=np.float32)
    counts = np.zeros((height, width), dtype=np.float32)

    for top in range(0, height, stride):
        bottom = min(top + patch_size, height)
        top = max(0, bottom - patch_size)

        for left in range(0, width, stride):
            right = min(left + patch_size, width)
            left = max(0, right - patch_size)

            occluded = image.copy()
            if occluded.ndim == 2:
                occluded[top:bottom, left:right] = fill_value
            else:
                occluded[top:bottom, left:right, :] = fill_value
            occluded_confidence = score_fn(occluded, predicted_label)
            importance = max(0.0, base_confidence - occluded_confidence)

            heatmap[top:bottom, left:right] += importance
            counts[top:bottom, left:right] += 1.0

    return np.divide(heatmap, counts, out=np.zeros_like(heatmap), where=counts > 0) # normalize by counts to get average importance per pixel


def reset_directory(output_dir: Path, clear_output_dir: bool) -> None:
    if clear_output_dir and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)


def save_classification_case_heatmaps(
    test_paths: list[Path],
    y_test: np.ndarray,
    y_pred: np.ndarray,
    confidences: np.ndarray,
    score_fn,
    model_input_loader,
    heatmap_config: HeatmapConfig,
) -> dict[str, list[dict[str, object]]]:
    """Generate and save occlusion heatmaps for TP, FP, FN, TN."""
    reset_directory(heatmap_config.heatmap_dir, heatmap_config.clear_output_dir)

    heatmap_results: dict[str, list[dict[str, object]]] = {}
    case_definitions = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "false_negative": (1, 0),
        "true_negative": (0, 0),
    }

    for case_name, (actual_label, predicted_label) in case_definitions.items():
        case_indices = [
            i
            for i, (actual, predicted) in enumerate(zip(y_test, y_pred))
            if actual == actual_label and predicted == predicted_label
        ]

        case_indices = sorted(
            case_indices,
            key=lambda i: confidences[i],
            reverse=True,
        )

        if heatmap_config.max_per_case is not None:
            case_indices = case_indices[:heatmap_config.max_per_case]

        case_dir = heatmap_config.heatmap_dir / case_name
        case_dir.mkdir(parents=True, exist_ok=True)

        heatmap_results[case_name] = []
        print(f"\nProcessing {case_name}: {len(case_indices)} images")

        for rank, idx in enumerate(case_indices):
            try:
                image_path = test_paths[idx]
                model_input_image = model_input_loader(image_path)
                display_image = read_display_image(image_path, heatmap_config.image_size)

                if heatmap_config.occlusion_fill_mode == "median":
                    fill_value = float(np.median(model_input_image))
                elif heatmap_config.occlusion_fill_mode == "zero":
                    fill_value = 0.0
                else:
                    raise ValueError("occlusion_fill_mode must be 'median' or 'zero'")

                heatmap = compute_occlusion_focus_heatmap(
                    model_input_image,
                    int(y_pred[idx]),
                    score_fn,
                    heatmap_config.occlusion_patch_size,
                    heatmap_config.occlusion_stride,
                    fill_value,
                )

                output_path = case_dir / f"{rank:03d}_{image_path.stem}_heatmap.png"
                title = (
                    f"{case_name.replace('_', ' ').title()} | "
                    f"actual {CLASS_NAMES[int(y_test[idx])]}, "
                    f"predicted {CLASS_NAMES[int(y_pred[idx])]} | "
                    f"conf {confidences[idx]:.3f}"
                )

                save_focus_pair(display_image, heatmap, title, output_path)

                result = {
                    "index": int(idx),
                    "rank": rank,
                    "actual_label": CLASS_NAMES[int(y_test[idx])],
                    "predicted_label": CLASS_NAMES[int(y_pred[idx])],
                    "source_image": str(image_path),
                    "heatmap": str(output_path),
                    "confidence": float(confidences[idx]),
                    "max_activation": float(np.max(heatmap)),
                }
                heatmap_results[case_name].append(result)
            except Exception as e:
                print(f"Failed on index {idx}: {e}")

        print(f"Done {case_name}: saved {len(heatmap_results[case_name])}")

    return heatmap_results

## XAI for Classical ML Model


### Approach

The classic pipeline does not operate directly on pixels. Each grayscale image is first converted into handcrafted features using the shared feature-extraction pipeline from `steelblast_classic_features.py`. The fitted SVM then produces a confidence score for a target label, and occlusion sensitivity measures how much that score drops when local image patches are masked. Regions that cause larger score drops are treated as more influential for the prediction.



### Parameters

- `heatmap_dir`: output folder for saved SVM heatmaps.
- `image_size=256`: image size used when loading grayscale images for feature extraction and occlusion analysis.
- `occlusion_patch_size=32`: side length of each masked square patch.
- `occlusion_stride=16`: step size between neighboring occlusion windows.
- `max_per_case=3`: maximum number of examples saved for each confusion-matrix case.
- `occlusion_fill_mode="median"`: fills the masked region with the image median intensity, which is less disruptive for grayscale texture images than zero-filling.
- `clear_output_dir=True`: removes previous outputs before writing a fresh set of heatmaps.


In [ ]:
import json
from datetime import datetime

from sklearn.pipeline import Pipeline
from steelblast_classic_features import (
    FeatureExtractionConfig,
    extract_features_from_image,
    read_grayscale_image,
)

heatmap_config = HeatmapConfig(heatmap_dir=
                Path("steelblast_svm_glcm_dwt_focus_heatmaps"), 
                               image_size=256, 
                               occlusion_patch_size=32,
                                 occlusion_stride=16, 
                                 max_per_case=5, 
                                 occlusion_fill_mode="median",
                                   clear_output_dir=True)

def save_heatmap_generation_metrics(
    heatmap_results: dict[str, list[dict[str, object]]],
    heatmap_config: HeatmapConfig,
    model_key: str,
    output_dir: Path = Path("."),
) -> Path:
    """Persist heatmap generation metadata for reproducibility and reporting."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = output_dir / f"heatmap_generation_metrics_{model_key}_{timestamp}.json"

    case_counts = {case: len(items) for case, items in heatmap_results.items()}
    per_case_stats: dict[str, dict[str, float | int]] = {}
    for case, items in heatmap_results.items():
        confidences = [float(item["confidence"]) for item in items]
        activations = [float(item["max_activation"]) for item in items]
        per_case_stats[case] = {
            "count": len(items),
            "mean_confidence": float(np.mean(confidences)) if confidences else 0.0,
            "mean_max_activation": float(np.mean(activations)) if activations else 0.0,
        }

    payload = {
        "model_key": model_key,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "heatmap_dir": str(heatmap_config.heatmap_dir),
        "config": {
            "image_size": heatmap_config.image_size,
            "occlusion_patch_size": heatmap_config.occlusion_patch_size,
            "occlusion_stride": heatmap_config.occlusion_stride,
            "max_per_case": heatmap_config.max_per_case,
            "occlusion_fill_mode": heatmap_config.occlusion_fill_mode,
            "clear_output_dir": heatmap_config.clear_output_dir,
        },
        "summary": {
            "total_saved": int(sum(case_counts.values())),
            "case_counts": case_counts,
            "per_case_stats": per_case_stats,
        },
        "results": heatmap_results,
    }

    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return output_path

def resolve_feature_adapter():
    """
    Build local image->feature adapter without requiring global `feature_config`.
    Uses default FeatureExtractionConfig if one is not present in notebook state.
    """
    %store -r feature_config
    if feature_config is None:
        cfg = FeatureExtractionConfig(
            image_size=256,
            illumination_normalization="clahe",
            glcm_levels=32,
            glcm_properties=(
                "contrast",
                "dissimilarity",
                "homogeneity",
                "energy",
                "correlation",
                "ASM",
            ),
            dwt_wavelet="db2",
            dwt_level=3,
        )
        print("feature_config not found in kernel state; using default FeatureExtractionConfig()")
    else:
        cfg = feature_config

    def image_to_features_fn(image: np.ndarray) -> np.ndarray:
        return extract_features_from_image(image, cfg)

    def grayscale_loader(image_path: Path, image_size: int) -> np.ndarray:
        return read_grayscale_image(image_path, image_size, cfg)

    return image_to_features_fn, grayscale_loader


def make_svm_score_fn(fitted_model: Pipeline, image_to_features_fn):
    """Return image->label confidence adapter for SVM on a probability-like 0..1 scale."""

    def _score_fn(image: np.ndarray, label: int) -> float:
        features = image_to_features_fn(image).reshape(1, -1)

        if hasattr(fitted_model, "predict_proba"):
            probabilities = fitted_model.predict_proba(features)[0]
            class_index = int(np.flatnonzero(fitted_model.classes_ == label)[0])
            return float(probabilities[class_index])

        scores = fitted_model.decision_function(features)
        if np.ndim(scores) == 1:
            positive_class = int(fitted_model.classes_[1])
            positive_probability = 1.0 / (1.0 + np.exp(-float(scores[0])))
            return positive_probability if label == positive_class else (1.0 - positive_probability)

        class_index = int(np.flatnonzero(fitted_model.classes_ == label)[0])
        row = np.asarray(scores[0], dtype=np.float64)
        shifted = row - np.max(row)
        probabilities = np.exp(shifted) / np.sum(np.exp(shifted))
        return float(probabilities[class_index])

    return _score_fn


def svm_predicted_label_confidences(
    fitted_model: Pipeline,
    features: np.ndarray,
    predictions: np.ndarray,
) -> np.ndarray:
    """Confidence per sample for its predicted class on a probability-like 0..1 scale."""
    if hasattr(fitted_model, "predict_proba"):
        probabilities = fitted_model.predict_proba(features)
        return np.asarray(
            [probabilities[index, np.flatnonzero(fitted_model.classes_ == label)[0]] for index, label in enumerate(predictions)],
            dtype=np.float64,
        )

    scores = fitted_model.decision_function(features)
    if np.ndim(scores) == 1:
        positive_class = int(fitted_model.classes_[1])
        positive_probabilities = 1.0 / (1.0 + np.exp(-np.asarray(scores, dtype=np.float64)))
        return np.where(predictions == positive_class, positive_probabilities, 1.0 - positive_probabilities)

    rows = np.asarray(scores, dtype=np.float64)
    shifted = rows - rows.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    return np.asarray(
        [probabilities[index, np.flatnonzero(fitted_model.classes_ == label)[0]] for index, label in enumerate(predictions)],
        dtype=np.float64,
    )


%store -r model_svm
fitted_model_svm = model_svm.best_estimator_ if hasattr(model_svm, "best_estimator_") else model_svm
%store -r X_test_svm
%store -r y_pred_svm
%store -r test_paths_svm
%store -r y_test_svm
image_to_features_fn, grayscale_loader = resolve_feature_adapter()
svm_confidences = svm_predicted_label_confidences(fitted_model_svm, X_test_svm, y_pred_svm)
svm_score_fn = make_svm_score_fn(fitted_model_svm, image_to_features_fn)
heatmap_paths = save_classification_case_heatmaps(
    test_paths_svm,
    y_test_svm,
    y_pred_svm,
    svm_confidences,
    svm_score_fn,
    lambda image_path: grayscale_loader(image_path, heatmap_config.image_size),
    heatmap_config,
)
svm_metrics_path = save_heatmap_generation_metrics(
    heatmap_paths,
    heatmap_config,
    model_key="svm_glcm_dwt",
)
print(f"Saved heatmap generation metrics: {svm_metrics_path}")

## XAI for Transfer Learning Model

### Approach

This section explains the occlusion-sensitivity workflow used for the ResNet50 transfer-learning classifier in the next cell. Unlike the classical pipeline, this model operates directly on RGB images. Each test image is loaded at the model input size, preprocessed with the ResNet50 preprocessing function, and passed through the network to obtain a class-1 score for `not-good`. Occlusion sensitivity then measures how much that score changes when local image patches are masked.



### Parameters

- `heatmap_dir`: output folder for saved transfer-learning heatmaps.
- `image_size=224`: input image size expected by the ResNet50 model.
- `occlusion_patch_size=32`: side length of each masked square patch.
- `occlusion_stride=16`: step size between neighboring occlusion windows.
- `max_per_case=3`: maximum number of examples saved for each confusion-matrix case.
- `occlusion_fill_mode="zero"`: fills masked patches with zeros, which is appropriate for the normalized RGB input used by the transfer model.
- `clear_output_dir=True`: removes previous outputs before writing a fresh set of heatmaps.


In [ ]:
from tensorflow import keras
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess


transfer_heatmap_config = HeatmapConfig(
    heatmap_dir=Path("steelblast_transfer_learning_occlusion_heatmaps"),
    image_size=224,
    occlusion_patch_size=32,
    occlusion_stride=16,
    max_per_case=5,
    occlusion_fill_mode="zero",
    clear_output_dir=True,
 )


def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)


def read_transfer_image(image_path: Path, image_size: int) -> np.ndarray:
    image = load_img(image_path, target_size=(image_size, image_size))
    return img_to_array(image).astype(np.float32)


def predict_transfer_score(
    image_rgb_float: np.ndarray,
    transfer_model: keras.Model,
    preprocess_fn,
 ) -> float:
    batch = preprocess_fn(image_rgb_float.copy())
    batch = np.expand_dims(batch, axis=0)
    return float(transfer_model.predict(batch, verbose=0).ravel()[0])


def make_transfer_score_fn(transfer_model: keras.Model, preprocess_fn):
    def _score_fn(image: np.ndarray, label: int) -> float:
        score = predict_transfer_score(image, transfer_model, preprocess_fn)
        return score if label == 1 else (1.0 - score)

    return _score_fn


def transfer_predicted_label_confidences(
    scores_class1: np.ndarray,
    predictions: np.ndarray,
) -> np.ndarray:
    return np.where(predictions == 1, scores_class1, 1.0 - scores_class1).astype(np.float64)


transfer_model_path = Path("steelblast_resnet50.h5")
transfer_dataset_dir = Path("data/SteelBlastQC")
#TODO replicate in the other approach 
transfer_model = keras.models.load_model(transfer_model_path)
transfer_test_paths, transfer_y_test = load_split(transfer_dataset_dir, "test")

transfer_scores_class1 = []
for image_path in transfer_test_paths:
    image_rgb = read_transfer_image(image_path, transfer_heatmap_config.image_size)
    transfer_scores_class1.append(
        predict_transfer_score(image_rgb, transfer_model, resnet_preprocess)
    )
transfer_scores_class1 = np.asarray(transfer_scores_class1, dtype=np.float32)
transfer_y_pred = (transfer_scores_class1 >= 0.5).astype(np.int64)

transfer_confidences = transfer_predicted_label_confidences(
    transfer_scores_class1,
    transfer_y_pred,
 )
transfer_score_fn = make_transfer_score_fn(transfer_model, resnet_preprocess)

transfer_heatmap_paths = save_classification_case_heatmaps(
    transfer_test_paths,
    transfer_y_test,
    transfer_y_pred,
    transfer_confidences,
    transfer_score_fn,
    lambda image_path: read_transfer_image(image_path, transfer_heatmap_config.image_size),
    transfer_heatmap_config,
 )
transfer_metrics_path = save_heatmap_generation_metrics(
    transfer_heatmap_paths,
    transfer_heatmap_config,
    model_key="transfer_resnet50",
)
print(f"Saved heatmap generation metrics: {transfer_metrics_path}")